In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import plotly.express as px
import plotly.graph_objects as go
import utils as utils

# pd.set_option('future.no_silent_downcasting', True)


# COMPLETE DATASETS

Save two options to create the whole dataset of the RAMA network, including atmospheric and pollutant data.
Also save the gap analisis with the following gaps description:

Consecutive nans | Replacement     | Description
-----------------|-----------------|----------------
x = 0               |   0   | No nan value
x = 1               |   1   | Missing value for a single hour
1 > x <= 3          |   2   | Missing values for less than 3 hours
3 > x <= 12         |   3   | Missing values for more than 3 hours but less or equal to 12
12 > x <= 24        |   4   | Missing values for more than 12 hours but less or equal to 24
24 > x <= 72        |   5   | Missing values for more than 1 day but less or equal to 3 days
72 > x <= 168       |   6   | Missing values for more than 3 days but less or equal to 7 days
168 > x <= 720      |   7   | Missing values for more than 7 days but less or equal to 30 days
720 > x <= 2,160    |   8   | Missing values for more than 30 days but less or equal to 90 days
2,160 > x <= 4,320  |   9   | Missing values for more than 90 days but less or equal to 180 days
4,320 > x           |  10   | Missing values for more than 180 days


By default, the datasets tables will have the columns:

* datetime
* year
* month
* day
* time
* dow
* variable
* station_1
* station_2
* ...
* station_n

## Hourly dataset RAMA-REDMET (1989-2025)

In [2]:
download_path = "../data/cdmx/raw"
output_path = "../data/datasets"

In [ ]:
main_folder = '../data/cdmx/raw/'
folders = ['RAMA','REDMET']

In [3]:

df = pd.DataFrame()
for folder in folders:
    print(main_folder + folder)
    files = os.listdir(main_folder + folder)
    files = [f for f in files if f.endswith('.xls')]
    
    for file in files:
        
        year = file[0:4]
        variable, _ = file[4::].lower().split('.')
        
        segment = pd.read_excel(f"{main_folder}{folder}/{file}")
        
        # avoid unnamed columns
        segment = segment.loc[:, ~segment.columns.str.contains('^Unnamed')]
        
        segment['variable'] = variable
        segment['year'] = int(year)
        
        df = pd.concat([df, segment], axis=0)
        
        
# Crea la columna fecha sumando la hora y la fecha
df['HORA'] = (df['HORA']-1).astype('str').str.zfill(2) + ':00:00'
df['datetime'] = pd.to_datetime(df['FECHA'].astype('str') + ' ' + df['HORA'], format='%Y-%m-%d %H:%M:%S')
df.drop(columns=['FECHA', 'HORA'], inplace=True)

df['time'] = df['datetime'].dt.hour.astype('int')
df['year'] = df['datetime'].dt.year
df['month'] = df['datetime'].dt.month
df['day'] = df['datetime'].dt.day
df['dow'] = df['datetime'].dt.dayofweek

df.replace("nr", np.nan, inplace=True)
df.replace(-99, np.nan, inplace = True)
df.reset_index(drop=True, inplace=True)
df.sort_values(by=['datetime'], inplace=True)
df.reset_index(drop=True, inplace=True)

not_station_columns = ['datetime', 'year', 'month', 'day', 'time', 'dow', 'variable']
station_columns = [col for col in df.columns if col not in not_station_columns]
df[station_columns] = df[station_columns].apply(pd.to_numeric, errors='coerce')
df = df[not_station_columns + station_columns]
df.reset_index(drop=True, inplace=True)

../data/cdmx/raw/RAMA
../data/cdmx/raw/REDMET


In [4]:
# Save the dataframe to a csv file
df.to_csv(output_path + '/ds_rama_redmet.csv', index=False)

In [ ]:
# bins = [0, 1, 3, 12, 24, 24*3, 24*7, 24*30, 24*30*3, 24*30*6, np.inf]
# labels = np.arange(1,len(bins))
# no_gap_value = 0

# df_temp_gaps = utils.get_data_gaps(df, bins = bins, labels = labels, 
#                                    no_gap_value = no_gap_value, format_gaps = int,
#                                    datetime_column = 'datetime', variable_column = 'variable',
#                                    aux_columns = ['year', 'month', 'day', 'time', 'dow'])

## Dataset RAMA 2 (2005 - today)

In [11]:
download_path = "../data/cdmx/raw/RAMA2"
output_path = "../data/datasets"

In [12]:
# read all files in the folder
files = os.listdir(download_path)
files = [f for f in files if f.endswith('.csv')]

In [13]:
variables = ['pm2', 'wdr', 'tmp', 'nox', 'no2', 'no', 'pm10', 'rh', 'co', 'so2', 'o3', 'wsp']
start_year = 2005
end_year = 2025

In [15]:
df = pd.DataFrame()
for file in files:
    # Divide file name to get the date, time and contaminant
    y, m, cont = file.split("_")
    cont = cont.split(".")[0]
    
    if (int(y) >= start_year) and (int(y) <= end_year) and (cont in variables):
        
        segment = pd.read_csv(f"{download_path}/{file}", skiprows=1)
        
        # avoid unnamed columns
        segment = segment.loc[:, ~segment.columns.str.contains('^Unnamed')]
        segment['Fecha'] = pd.to_datetime(segment['Fecha'], format='%d-%m-%Y')
        
        segment['variable'] = cont
        segment['year'] = segment['Fecha'].dt.year
        segment['month'] = segment['Fecha'].dt.month
        segment['day'] = segment['Fecha'].dt.day
        segment['dow'] = segment['Fecha'].dt.dayofweek
        
        df = pd.concat([df, segment], axis=0)
        
# Create time and datetime columns
df['time'] = (df['Hora']-1).astype('int')
df['datetime'] = df['Fecha'] + pd.to_timedelta(df['time'].astype('str').str.zfill(2) + ':00:00')
df.drop(columns=['Fecha','Hora'], inplace=True)

df.replace("nr", np.nan, inplace=True)
df.reset_index(drop=True, inplace=True)
df.sort_values(by=['datetime'], inplace=True)

not_station_columns = ['datetime', 'year', 'month', 'day', 'time', 'dow', 'variable']
station_columns = [col for col in df.columns if col not in not_station_columns]
df[station_columns] = df[station_columns].apply(pd.to_numeric, errors='coerce')
df = df[not_station_columns + station_columns]
df.reset_index(drop=True, inplace=True)

In [16]:
# Save the dataframe to a csv file
df.to_csv(output_path + '/ds_rama2.csv', index=False)

In [ ]:
# bins = [0, 1, 3, 12, 24, 24*3, 24*7, 24*30, 24*30*3, 24*30*6, np.inf]
# labels = np.arange(1,len(bins))
# no_gap_value = 0

# df_temp_gaps = utils.get_data_gaps(df, bins = bins, labels = labels, 
#                                    no_gap_value = no_gap_value, format_gaps = int,
#                                    datetime_column = 'datetime', variable_column = 'variable',
#                                    aux_columns = ['year', 'month', 'day', 'time', 'dow'])

# df_temp_gaps.to_csv(output_path + '/ds_rama2_md.csv', index=False)


# Curated dataset

Based on the analysis of the previous sources, it was decided to use 

In [5]:
download_path = "../data/cdmx/raw"
output_path = "../data/datasets"

In [6]:
catalog_stations = pd.read_csv('../data/cdmx/curated/cat_stations.csv')
active_stations = catalog_stations.query('status == "active"')['key'].tolist()
not_station_columns = ['datetime', 'year', 'month', 'day', 'time', 'dow', 'variable']

In [7]:
df1 = pd.read_csv(output_path + '/ds_rama_redmet.csv', parse_dates=['datetime'])
df2 = pd.read_csv(output_path + '/ds_rama2.csv', parse_dates=['datetime'])

# df1.set_index('datetime', inplace = True)
# df2.set_index('datetime', inplace = True)

In [83]:
cols2 = df2.query('datetime >= "2012-01-01"').dropna(axis=1, how='all').columns
cols1 = df1.query('datetime >= "2012-01-01"').dropna(axis=1, how='all').columns

stations1 = set([c for c in cols1 if c in active_stations])
stations2 = set([c for c in cols2 if c in active_stations])
stations_selected = list(stations1.intersection(stations2))

In [84]:
start_year = 2012
end_year = 2026

In [ ]:
dataset = df1.query(f'datetime >= {start_year} and datetime < {end_year}  ')
dataset.query(f'variable.str.isin({variables})')

In [100]:
variables = ['pm10',  'wsp',   'no',
             'rh',  'nox',  'no2',
             'co',   'o3', 'pm25',
             'tmp',  'wdr',  'so2']

In [101]:
dataset.query(f'variable.isin({variables})')['variable'].value_counts()

variable
wsp     116848
no      116848
rh      116848
nox     116848
no2     116848
co      116848
o3      116848
tmp     116848
wdr     116848
so2     116848
pm10    108072
pm25    108072
Name: count, dtype: int64

In [87]:
dataset[not_station_columns + stations_selected]

,datetime,year,month,day,time,dow,variable,ACO,SJA,XAL,...,AJM,AJU,COY,GAM,MER,UIZ,TAH,UAX,SAG,INN
2366688,2012-01-01 00:00:00,2012,1,1,0,6,pm10,113.0,NaN,100.0,...,NaN,NaN,NaN,NaN,90.0,124.0,119.0,NaN,915.0,NaN
2366689,2012-01-01 00:00:00,2012,1,1,0,6,wsp,1.5,NaN,NaN,...,NaN,NaN,NaN,NaN,1.4,NaN,NaN,NaN,1.2,NaN
2366690,2012-01-01 00:00:00,2012,1,1,0,6,no,7.0,83.0,31.0,...,NaN,NaN,4.0,NaN,48.0,24.0,9.0,NaN,34.0,NaN
2366691,2012-01-01 00:00:00,2012,1,1,0,6,rh,77.0,NaN,63.0,...,NaN,NaN,NaN,NaN,64.0,NaN,79.0,NaN,71.0,NaN
2366692,2012-01-01 00:00:00,2012,1,1,0,6,nox,38.0,129.0,78.0,...,NaN,NaN,41.0,NaN,93.0,65.0,44.0,NaN,78.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3868155,2025-04-30 23:00:00,2025,4,30,23,2,rh,NaN,NaN,NaN,...,63.0,NaN,NaN,58.0,48.0,53.0,49.0,50.0,NaN,78.0
3868156,2025-04-30 23:00:00,2025,4,30,23,2,pm10,20.0,NaN,NaN,...,26.0,NaN,NaN,28.0,21.0,23.0,39.0,NaN,NaN,34.0
3868157,2025-04-30 23:00:00,2025,4,30,23,2,pm25,NaN,NaN,NaN,...,16.0,NaN,NaN,16.0,15.0,15.0,NaN,15.0,NaN,18.0
3868158,2025-04-30 23:00:00,2025,4,30,23,2,no,2.0,NaN,11.0,...,0.0,NaN,NaN,NaN,2.0,2.0,1.0,1.0,1.0,NaN
